# 05 · Logistic Regression
> **Part:** Part 1 · Classical supervised learning  
> **Family:** `Supervised · Classification (linear)`

## Definition
Despite the name, logistic regression is a *classifier*. It computes a linear score z = w·x + b and squashes it through
the sigmoid σ(z) = 1/(1+e⁻ᶻ) to get a probability P(y=1 | x). The decision boundary (P = 0.5) is the straight line w·x + b = 0.

## Intuition
Linear regression extended to probabilities. Far on one side of the line the model is confident (P≈0 or 1); near
the line it is unsure (P≈0.5). Training pushes the line to put defaulters on one side and good borrowers on the other,
punishing *confident mistakes* very hard.


## The Math

```text
P(y=1|x) = σ(w·x + b)
    Loss (binary cross-entropy / negative log-likelihood):  L = −1/n Σ [ y log p + (1−y) log(1−p) ]
    Gradient (beautifully simple):  ∂L/∂w = 1/n Σ (pᵢ − yᵢ) xᵢ ,   ∂L/∂b = 1/n Σ (pᵢ − yᵢ)
    Multi-class: softmax instead of sigmoid.  Regularize with an L2/L1 penalty exactly like lesson 04.
```


In [ ]:
# Setup, Imports & Styling
%matplotlib inline
import os
import pathlib
import sys
from IPython.display import Image, display

# Ensure parent folder is in path to import mlatlas
parent_dir = pathlib.Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mlatlas import data, viz, cli

# Ensure interactive inline mode and configure Seaborn theme
plt.ion()
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 100, "figure.facecolor": "white"})


### Step 1: Exploratory Classification Data Visualization
We visualize a synthetic dataset of loan applicants with income and credit debt ratio features, labeled as defaulted (1) or repaid (0).


In [ ]:
X, y, names = data.loan_default(n=300, seed=0)

plt.figure(figsize=(8, 5))
sns.scatterplot(x=X[:, 0], y=X[:, 1], hue=y, palette=["#4c78a8", "#e45756"], s=60, alpha=0.9, edgecolor="w")
plt.title("Loan Applicants: Income vs. Debt Ratio", fontsize=12, fontweight="bold")
plt.xlabel("Income ($1,000s/yr)")
plt.ylabel("Debt Ratio (%)")
plt.legend(title="Default (1=Yes)", loc="upper right")
plt.tight_layout()
plt.show()


### Step 2: From-Scratch Algorithm Implementation
Below is the complete implementation of the model components in pure NumPy.


In [ ]:
# Main Algorithm & Training Routine
def main(gif, quick):
    X, y, names = data.loan_default(400, seed=2); Z, mu, sd = data.standardize(X)
    rng = np.random.default_rng(3); w, b = rng.normal(0, 1.5, 2), 0.; lr = .4; steps = 80 if quick else 300; H = []
    for _ in range(steps + 1):
        p = 1 / (1 + np.exp(-(Z @ w + b)))
        H.append((w.copy(), b, -np.mean(y * np.log(p + 1e-9) + (1 - y) * np.log(1 - p + 1e-9)), ((p > .5) == y).mean()))
        w = w - lr * Z.T @ (p - y) / len(y); b = b - lr * (p - y).mean()
    print(f"  final loss {H[-1][2]:.3f}, accuracy {H[-1][3]:.3f}; weights (standardized) {np.round(H[-1][0], 2)}")
    xx, yy, G = viz.grid(X, pad=5); Gz = (G - mu) / sd
    fig, (a1, a2) = viz.subplots(1, 2, w=11, h=4.4); idx = viz.geo(len(H), 12 if quick else 55)
    loss = [h[2] for h in H]

    def update(i):
        a1.clear(); a2.clear(); w_, b_, l, acc = H[i]
        P = 1 / (1 + np.exp(-(Gz @ w_ + b_))); viz.show_map(a1, xx, yy, P, prob=True)
        a1.contour(xx, yy, P.reshape(xx.shape), levels=[.5], colors="k", linewidths=2); viz.scatter(a1, X, y)
        a1.set(title=f"iteration {i}: accuracy {acc:.0%}", xlabel=names[0], ylabel=names[1], xlim=(xx.min(), xx.max()), ylim=(yy.min(), yy.max()))
        a2.plot(loss, color=viz.PALETTE[3]); a2.scatter(i, loss[i], color="k", zorder=5); a2.set(title="Cross-entropy loss", xlabel="iteration", ylim=(0, max(loss) * 1.05))
    viz.animate(fig, update, idx, gif, fps=8)


### Step 3: Decision Boundary & Probability Surface
We evaluate the model by computing the decision boundary (where $P(y=1|x) = 0.5$) and plotting the sigmoid probability landscape.


In [ ]:
Z, mu, sd = data.standardize(X)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -25, 25)))

w = np.zeros(2)
b = 0.0
lr = 0.1
for _ in range(800):
    p = sigmoid(Z @ w + b)
    grad_w = Z.T @ (p - y) / len(y)
    grad_b = np.mean(p - y)
    w -= lr * grad_w
    b -= lr * grad_b

# Create meshgrid for probability contour
x_min, x_max = Z[:, 0].min() - 0.5, Z[:, 0].max() + 0.5
y_min, y_max = Z[:, 1].min() - 0.5, Z[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 150), np.linspace(y_min, y_max, 150))
grid_points = np.c_[xx.ravel(), yy.ravel()]
probs = sigmoid(grid_points @ w + b).reshape(xx.shape)

plt.figure(figsize=(9, 5.5))
contour = plt.contourf(xx, yy, probs, levels=25, cmap="RdBu_r", alpha=0.75)
plt.colorbar(contour, label="P(Default = 1)")
plt.contour(xx, yy, probs, levels=[0.5], colors="black", linewidths=2.5, linestyles="--")
sns.scatterplot(x=Z[:, 0], y=Z[:, 1], hue=y, palette=["#4c78a8", "#e45756"], edgecolor="k", s=55, zorder=5)
plt.title("Logistic Regression: Probability Surface & Decision Boundary (P = 0.5)", fontsize=12, fontweight="bold")
plt.xlabel("Standardized Income")
plt.ylabel("Standardized Debt Ratio")
plt.tight_layout()
plt.show()


### Step 4: Industry Benchmark (Comparison with scikit-learn)
We compare against `sklearn.linear_model.LogisticRegression`.


In [ ]:
from sklearn.linear_model import LogisticRegression

sk_clf = LogisticRegression(penalty=None, solver="lbfgs").fit(Z, y)
print("--- Model Parameters Comparison ---")
print(f"NumPy Scratch: w = {w.round(4)}, b = {b:.4f}")
print(f"scikit-learn:  w = {sk_clf.coef_[0].round(4)}, b = {sk_clf.intercept_[0]:.4f}")


### Step 5: Visualizing Learning Dynamics

Loan-default data (income vs debt ratio). Watch the probability map (blue = repay, orange = default) and the black
P=0.5 decision line rotate from a random starting guess to the best separator; right: loss falls, accuracy rises.


In [ ]:
# Display the learning animation inline
media_file = f"../outputs/05_logistic_regression.gif"
if os.path.exists(media_file):
    display(Image(filename=media_file))
elif os.path.exists(f"outputs/05_logistic_regression.gif"):
    display(Image(filename=f"outputs/05_logistic_regression.gif"))


## Strengths & weaknesses

- Fast, convex (a single global optimum), well-calibrated probabilities, interpretable (odds ratio = e^w).
- Only linear boundaries unless you add polynomial/interaction features; needs scaled inputs for gradient descent.


## 🧠 Check Your Understanding

<details>
<summary><b>1. Why can't we use Mean Squared Error (MSE) directly with Logistic Regression?</b></summary>
Because passing a linear combination through the non-linear sigmoid function turns the MSE loss surface into a non-convex function with multiple local minima. Binary Cross-Entropy (Log Loss) guarantees a convex surface with a unique global minimum.
</details>

<details>
<summary><b>2. What is the geometric interpretation of the decision boundary?</b></summary>
The decision boundary is the hyperplane defined by $w^T x + b = 0$, where the sigmoid output $\sigma(0) = 0.5$. On one side, predicted probability is $> 0.5$; on the other, $< 0.5$.
</details>


## 🧪 Guided Experiments

- Add `debt_ratio**2` as a third feature (needs a 3-D weight vector) and see a curved boundary appear in the original space.
- Change the learning rate: where does training become unstable?
